# COMBINED MCMC ANALYSIS: CMB + BAO + SNe + H(z)**Author**: Ricardo Alvim**Date**: January 2026**Objective**: Perform the final parameter constraints for Paper I by combining all datasets.---## Datasets Used1.  **CMB**: Planck 2018 (Compressed Likelihood / Shift Parameters)2.  **SNe**: Pantheon+ (Type Ia Supernovae)3.  **BAO**: SDSS DR12 + DR16 (Baryon Acoustic Oscillations)4.  **H(z)**: Cosmic Chronometers## Model: Evaporating UniverseParameters to constrain:- **H0**: Hubble Constant- **Omega_m**: Matter Density- **w0**: Dark Energy Equation of State (Today)- **z_trans**: Transition Redshift (Optional/Fixed)

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.integrate import quadfrom scipy.optimize import minimizeimport emceeimport cornerimport jsonfrom datetime import datetimefrom multiprocessing import Poolimport timeplt.rcParams.update({'font.size': 12, 'figure.dpi': 150})print("="*70)print("COMBINED MCMC: EVAPORATING UNIVERSE (High-Performance Mode)")print("="*70)

In [ ]:
# =============================================================# 1. DATASETS (EMBEDDED FOR ROBUSTNESS)# =============================================================# --- BAO DATA (SDSS DR12) ---# z_eff, D_V(z)bao_data = np.array([[0.38, 1477.0, 16.0],  # BOSS DR12[0.51, 1877.0, 19.0],  # BOSS DR12[0.61, 2140.0, 22.0]   # BOSS DR12])rd_fiducial = 147.78 # Mpc# --- SNe DATA (Pantheon+ - Binned) ---# Simplification: Use binned Pantheon data for speed# z, m_b, errorsne_data = np.array([[0.014, 14.57, 0.15],[0.026, 15.96, 0.12],[0.036, 16.63, 0.10],[0.100, 19.10, 0.08],[0.200, 20.84, 0.08],[0.300, 21.90, 0.09],[0.400, 22.65, 0.09],[0.500, 23.25, 0.10],[0.700, 24.10, 0.12],[1.000, 24.95, 0.15],[1.500, 25.80, 0.20]])# --- Cosmic Chronometers (H(z)) ---# z, H(z), errorhz_data = np.array([[0.070, 69.0, 19.6],[0.120, 68.6, 26.2],[0.200, 72.9, 29.6],[0.280, 88.8, 36.6],[0.380, 83.0, 13.5],[0.480, 97.0, 62.0],[0.593, 104.0, 13.0],[0.781, 105.0, 12.0],[0.900, 117.0, 23.0],[1.037, 154.0, 20.0],[1.363, 160.0, 33.6],[1.750, 202.0, 40.0],[1.965, 186.5, 50.4]])# --- CMB Compressed (Planck 2018) ---# Shift parameter R, Acoustic Scale l_a, Omega_b*h^2# Values and Covariancecmb_mean = np.array([1.74963, 301.808, 0.02237])cmb_cov = np.array([[1.6e-5, 1.3e-3, -1.8e-7],[1.3e-3, 0.11, -5.7e-5],[-1.8e-7, -5.7e-5, 2.3e-8]])cmb_inv_cov = np.linalg.inv(cmb_cov)print("Datasets Loaded Successfully.")

In [ ]:
# =============================================================# 2. THEORY FUNCTIONS# =============================================================def hubble_normalized(z, Om0, w0, z_trans=0.22):# Simplified for speed: we assume transition is handled or we use a phenomenological w(z)# For Combined fit, we test simple CPL or Evaporating effective model# Evaporating Model: w(z) evolution# rho_de / rho_de0# Let's use the explicit integral form for robustness# Model A: w(z) = w0 (Constant w model check)# Model B: w(z) = w0 + wa * z/(1+z) (CPL)# Model C: Evaporating (Sigmoid w_late -> w_early)# Here, let's implement the Evaporating parametrization# w(z) = w_late + (w_early - w_late) * sigmoidw_late = -1.2w_early = -0.05width = 0.4# This is expensive for integration inside MCMC loop# Use approximation for E(z) if possible, or fast integration# To speed up MCMC, let's assume effective constant w0 behavior for broad evolution# OR use w0 as the LOW REDSHIFT EFFECTIVE PARAMETER which is what we constrainOl0 = 1 - Om0E2 = Om0 * (1+z)**3 + Ol0 * (1+z)**(3*(1+w0))return np.sqrt(E2)def comoving_dist(z, h, Om0, w0):c = 299792.458integrand = lambda zp: 1.0 / hubble_normalized(zp, Om0, w0)if np.ndim(z) == 0:d_c, _ = quad(integrand, 0, z)else:d_c = np.array([quad(integrand, 0, zi)[0] for zi in z])return (c / (100*h)) * d_cdef angular_diameter_dist(z, h, Om0, w0):return comoving_dist(z, h, Om0, w0) / (1+z)def sound_horizon(h, Om0, Ob0):# Approximate formula for r_s(z_drag)# Use fitting function by Aubourg et al. 2015# Simplified for standard rangeswm = Om0 * h**2wb = Ob0 * h**2rs = 55.154 * np.exp(-72.3 * (wm + 0.0006)**2) / (wm**0.25321 * wb**0.12807)return rs

In [ ]:
# =============================================================# 3. LIKELIHOODS# =============================================================def log_likelihood(theta):h, Om0, w0 = theta# Priors (Flat)if not (0.5 < h < 0.9 and 0.1 < Om0 < 0.5 and -2.0 < w0 < -0.3):return -np.inf# Calculate derived params# Ob0 fixed or marginalized? Let's fix Ob0*h^2 = 0.0224 for geometryObh2 = 0.02237Ob0 = Obh2 / h**2H0 = 100 * hchi2 = 0.0# --- 1. SNe Likelihood ---z_sne = sne_data[:, 0]mb_obs = sne_data[:, 1]mb_err = sne_data[:, 2]dL = (1+z_sne) * comoving_dist(z_sne, h, Om0, w0)mu_theo = 5 * np.log10(dL) + 25# Marginalize over M (absolute magnitude nuisance)# delta = mu_theo - mb_obs# A = sum(delta^2/sig^2), B = sum(delta/sig^2), C = sum(1/sig^2)# min_chi2 = A - B^2/Cdiff = mu_theo - mb_obsinv_var = 1.0 / mb_err**2A = np.sum(diff**2 * inv_var)B = np.sum(diff * inv_var)C = np.sum(inv_var)chi2_sne = A - B**2 / Cchi2 += chi2_sne# --- 2. H(z) Likelihood ---z_hz = hz_data[:, 0]H_obs = hz_data[:, 1]H_err = hz_data[:, 2]H_theo = 100 * h * hubble_normalized(z_hz, Om0, w0)chi2_hz = np.sum(((H_theo - H_obs) / H_err)**2)chi2 += chi2_hz# --- 3. BAO Likelihood ---z_bao = bao_data[:, 0]DV_obs = bao_data[:, 1]DV_err = bao_data[:, 2]rs = sound_horizon(h, Om0, Ob0)for i in range(len(z_bao)):z = z_bao[i]dc = comoving_dist(z, h, Om0, w0)Hz = 100 * h * hubble_normalized(z, Om0, w0)c = 299792.458DV_theo = (z * dc**2 * c / Hz)**(1/3)# Measurement is usually DV/rs or DV * (rd_fid/rs)# Here standard form: alpha = (DV/rs)/(DV_fid/rd_fid)# Let's assume direct DV calibration# Actually BAO obs are often DV(z)# Correct scaling: DV_obs * (rs_fid / rs_true)residual = DV_obs * (rd_fiducial / rs) - DV_theochi2 += (residual / DV_err)**2# --- 4. CMB Likelihood (Compressed) ---# Shift R = sqrt(Om0 H0^2) * r(z_star) / cz_star = 1090.0da_star = angular_diameter_dist(z_star, h, Om0, w0)# R parameterR = np.sqrt(Om0) * H0 / 299792.458 * (da_star * (1+z_star))# Acoustic scale la = pi * da(z_star) / rs# rs integral needs radiation... approximation used abovela = np.pi * da_star / rs# Vector x = [R, la, Obh2]# Obh2 is derived from paramsx = np.array([R, la, Obh2])delta_x = x - cmb_meanchi2_cmb = delta_x.T @ cmb_inv_cov @ delta_xchi2 += chi2_cmbreturn -0.5 * chi2

In [ ]:
# =============================================================# 4. RUN MCMC (CONFIGURATION & PARALLEL EXECUTION)# =============================================================# CONFIGURATIONQUICK_TEST = False  # Set to True for a quick debug run (100 steps)USE_MULTIPROCESSING = Truenwalkers = 32ndim = 3nsteps = 100 if QUICK_TEST else 5000print(f"--- MCMC CONFIGURATION ---")print(f"Mode:    {'⚡ QUICK TEST' if QUICK_TEST else '🐢 FULL PAPER RUN'}")print(f"Steps:   {nsteps}")print(f"Walkers: {nwalkers}")print(f"Multiprocessing: {USE_MULTIPROCESSING}")print(f"--------------------------")# Initial guess [h, Om0, w0]# We start close to the expected values to help convergenceinitial = np.array([0.73, 0.30, -1.1])pos = initial + 1e-3 * np.random.randn(nwalkers, ndim)# Execution Logicif __name__ == "__main__":t0 = time.time()if USE_MULTIPROCESSING:with Pool() as pool:sampler = emcee.EnsembleSampler(nwalkers, ndim, log_likelihood, pool=pool)print("Starting MCMC (Parallelized)...")sampler.run_mcmc(pos, nsteps, progress=True)else:sampler = emcee.EnsembleSampler(nwalkers, ndim, log_likelihood)print("Starting MCMC (Single Core)...")sampler.run_mcmc(pos, nsteps, progress=True)t1 = time.time()print(f"MCMC Finished in {(t1-t0)/60:.1f} minutes!")

In [ ]:
# =============================================================# 5. RESULTS# =============================================================discard_len = 50 if QUICK_TEST else 1000flat_samples = sampler.get_chain(discard=discard_len, flat=True)print(f"Total samples: {flat_samples.shape[0]}")# Mean and 1-sigmalabels = ["h", "Ω_m", "w_0"]for i in range(ndim):mcmc = np.percentile(flat_samples[:, i], [16, 50, 84])q = np.diff(mcmc)print(f"{labels[i]} = {mcmc[1]:.4f} (+{q[1]:.4f} / -{q[0]:.4f})")# Corner Plotfig = corner.corner(flat_samples,labels=labels,truths=[0.733, 0.30, -1.13],quantiles=[0.16, 0.5, 0.84],show_titles=True)plt.savefig('combined_corner.png')plt.show()# Save Results to JSONresults = {"dataset": "COMBINED (CMB+SNe+BAO+Hz)","run_type": "QUICK_TEST" if QUICK_TEST else "FULL_RUN","h": {"mean": float(np.mean(flat_samples[:,0])),"err": float(np.std(flat_samples[:,0]))},"Om0": {"mean": float(np.mean(flat_samples[:,1])),"err": float(np.std(flat_samples[:,1]))},"w0": {"mean": float(np.mean(flat_samples[:,2])),"err": float(np.std(flat_samples[:,2]))}}with open('mcmc_combined_results.json', 'w') as f:json.dump(results, f, indent=2)try:from google.colab import filesfiles.download('combined_corner.png')files.download('mcmc_combined_results.json')except:pass